In [14]:

"""
Balanced Oil and Vinegar Şemasına Yönelik Kipnis-Shamir Saldırısı
(Çift Karakteristik Durumu: q = 2^k)
====================================================================

Bu modül, bir önceki (q tek) sürümde ele alınan Kipnis-Shamir
saldırısının, ÇİFT KARAKTERİSTİKLİ (q = 2^k) sonlu cisimler üzerinde
tanımlı Balanced Oil and Vinegar şemasına uygulanan öğretici SageMath
simülasyonunu içermektedir.

Neden Ayrı Bir Sürüm Gerekiyor?
---------------------------------
q tek karakteristikli durumda saldırı, W = W1^{-1} * W2 matrisinin
karakteristik polinomunun C(lambda) = C1(lambda)^2 biçiminde TAM KARE
olmasına ve bu polinomun kareköküne (C1) dayanıyordu. Karakteristik 2
(q = 2^k) durumunda ise iki temel fark ortaya çıkar:

1. Kuadratik Form Temsili: Karakteristik 2'de x_i^2 terimi, x_i*x_i
   çarpımının "kutuplaşmış" (polarize) hali ile AYNI DEĞİLDİR; yani
   simetrik bilineer form B(x, y) = Q(x+y) - Q(x) - Q(y) tekniği
   karakteristik 2'de dejenere olur (çünkü 2 = 0). Bu nedenle kuadratik
   form matrisleri bu kodda ÜST ÜÇGENSEL (P_k) olarak saklanır ve
   simetrik bilineer form Q_bar_k = P_k + P_k^T ayrı olarak hesaplanır;
   saldırı W1, W2 kombinasyonlarını P_k değil Q_bar_k matrisleri
   üzerinden kurar.

2. Karakteristik Polinomun Kareköpü: Karakteristik 2'de, çift dereceli
   katsayılardan oluşan bir polinomun (yani C(lambda) = C1(lambda)^2
   formundaki bir polinomun) kareköpü, "Freshman's Dream" özdeşliği
   (a + b)^2 = a^2 + b^2 sayesinde KOLAYCA hesaplanabilir: C(lambda)'nın
   yalnızca çift indisli katsayıları alınıp bu katsayıların (GF(2^k)
   içinde her elemanın tek bir karekökü olduğundan) kareköpü alınarak
   C1(lambda) doğrudan inşa edilir. Tek indisli katsayıların sıfır
   olması, C(lambda)'nın gerçekten tam kare olduğunun bir ön koşuludur.

3. Özuzay ve Aday Seçimi: q tek durumunda C1(W)'nin çekirdeği doğrudan
   Oil uzayı adayını verirken, çift karakteristik durumunda literatürde
   (bkz. tez Algoritma 5.2 kaynağı) katlılığı (multiplicity) 1 olan bir
   DOĞRUSAL kökün (lambda) özuzayından iki bağımsız "tohum" vektör (v1,
   v2) seçilip, bu tohumların rastgele W1, W2 kombinasyonlarıyla adım
   adım GENİŞLETİLMESİ (rastgele matrisler altında görüntülerinin
   uzaya eklenmesi) yoluyla Oil uzayı yeniden inşa edilir. Bu, tek
   karakteristik sürümündeki "aday vektörleri açık anahtarla test etme"
   yaklaşımından farklı, fakat aynı temel doğrusal cebirsel gözleme
   (gizli Oil uzayının bir değişmez alt uzay olması) dayanan alternatif
   bir kurtarma stratejisidir.

4. Cisim İnişi (Field Descent): İmza sahtekarlığı (forgery) aşamasında,
   GF(2^k) üzerindeki Oil değişkenlerine göre denklemler artık DOĞRUSAL
   DEĞİL, KUADRATİKTİR (çünkü F_oo bloğunun köşegeni sıfır olmak
   ZORUNDA DEĞİLDİR; yalnızca Oil x Oil ÇAPRAZ terimleri, yani i != j
   için M_k[i,j], sıfırdır -- x_i^2 terimleri sıfır olmak zorunda
   değildir). Ancak GF(2^k) üzerinde Frobenius otomorfizması x -> x^2
   DOĞRUSAL bir dönüşüm olduğundan, her bir GF(2^k) denklemi, temel
   cisim GF(2) üzerinde tanımlı DOĞRUSAL bir denklem sistemine
   "indirgenebilir" (bu teknik literatürde "field descent" veya "Weil
   descent" olarak bilinir). Bu kodda _to_base_vector metodu, GF(2^k)
   elemanlarını GF(2) üzerindeki koordinat vektörlerine çevirerek bu
   indirgemeyi gerçekleştirir; elde edilen GF(2) üzerindeki doğrusal
   sistem çözülüp katsayılar GF(2^k)'ya geri taşınarak Oil değişkenleri
   bulunur.

Bu Simülasyonun Sınırları ve Amaçları
---------------------------------------
- Bu kod GÜVENLİ parametreler üretmek için değil, saldırının
  karakteristik 2 durumundaki cebirsel özelliklerini (kuadratik form
  temsili, polinom kareköpü, cisim inişi) somut biçimde göstermek
  amacıyla hazırlanmıştır.
- q'nun ÇİFT (q = 2^k biçiminde) olması ZORUNLU bir ön koşuldur; bu
  koşul __init__ içinde açıkça denetlenir.
- Öğretici sadelik için v = o (balanced OV) seçilmiştir.
- Secret_Oil_Space, gerçek bir saldırganın erişemeyeceği bir bilgidir
  ve kodda YALNIZCA simülasyon doğrulaması amacıyla saklanır.

Algoritmanın Genel Akışı
-------------------------
1. Sistem İnşası (generate_keys):
   Gizli T dönüşümü, gizli Oil uzayı ve merkezi dönüşüm matrisleri
   (üst üçgensel P_k ve bunların simetrik hali Q_bar_k) üretilir;
   bunlardan açık anahtar polinomları (P_polys) türetilir.

2. Alt Uzay Kurtarma (_attempt_subspace_recovery):
   Açık anahtar matrislerinin (Q_bar_k) rastgele doğrusal
   kombinasyonlarından W1 (tersinir) ve W2 oluşturulur; W = W1^{-1} *
   W2'nin karakteristik polinomunun karakteristik-2 kareköpü alınarak
   katlılığı 1 olan bir kökün özuzayından tohum vektörler seçilir ve
   bu tohumlar adım adım genişletilerek Oil uzayı adayı inşa edilir.

3. Saldırı (run_attack):
   Alt uzay kurtarma denemesi (retry mekanizmasıyla) tekrarlanır; bir
   Oil uzayı bulunduğunda denk bir (T', F') anahtar çifti kurulur ve
   cisim inişi tekniğiyle rastgele bir mesaj için sahte imza (forgery)
   üretilip orijinal açık anahtarla doğrulanır.

Bu implementasyon; yüksek lisans tezi kapsamında Kipnis-Shamir
saldırısının karakteristik 2 özelindeki cebirsel inceliklerini
somutlaştırmak amacıyla eğitim/araştırma amaçlı geliştirilmiştir.

Bağımlılıklar
-------------
Bu kod SageMath ortamında çalıştırılmak üzere tasarlanmıştır ve sonlu
cisimler (GF), çok değişkenli polinom halkaları (PolynomialRing), vektör
uzayları (VectorSpace), matris cebiri, karakteristik polinom
çarpanlarına ayırma ve cisim inişi (field descent) için koordinat
vektörü dönüşümleri gibi SageMath'in yerleşik simgesel/sayısal cebir
araçlarına dayanır.


Referans: Tez, Bölüm 6
"""

from sage.all import *

class KipnisShamirEvenCharacteristicDemo:
    """
    Balanced Oil and Vinegar şemasına yönelik Kipnis-Shamir saldırısının,
    ÇİFT KARAKTERİSTİKLİ (q = 2^k) sonlu cisimler için uyarlanmış
    öğretici SageMath simülasyonu.

    Bu sınıf, q tek karakteristik sürümünden farklı olarak, karakteristik
    2'nin kendine özgü cebirsel özelliklerini (kuadratik form/simetrik
    bilineer form ayrımı, polinom kareköpü, Frobenius tabanlı cisim
    inişi) hesaba katan bir saldırı zinciri uygular.

    Matematiksel Kurulum
    ---------------------
    - F_q = GF(2^k) : Temel sonlu cisim; q ZORUNLU olarak çift
      (2'nin kuvveti) olmalıdır.
    - base_F = GF(2) : F_q'nun asal alt cismi; cisim inişi (field
      descent) tekniğinde imza denklemlerinin indirgeneceği temel
      cisimdir.
    - ext_degree : F_q'nun GF(2) üzerindeki genişleme derecesi (k);
      her F_q elemanının GF(2) üzerinde kaç koordinatla temsil
      edildiğini belirler.
    - R = F_q[x_0, ..., x_{n-1}] : n = v + o = 2o değişkenli, F_q
      katsayılı çok değişkenli polinom halkası.

    Parametreler
    ------------
    q : int
        Temel sonlu cismin eleman sayısı; q = 2^k biçiminde ÇİFT
        olmalıdır (aksi halde ValueError fırlatılır).
    o : int
        Oil değişkenlerinin sayısı. Balanced OV varsayımı gereği
        Vinegar değişken sayısı da o'ya eşit alınır (v = o).

    Öznitelikler (Attributes)
    --------------------------
    v : int
        Vinegar değişkenlerinin sayısı (v = o, balanced OV).
    n : int
        Toplam değişken sayısı (n = o + v = 2o).
    F : FiniteField
        Temel sonlu cisim F_q = GF(2^k).
    base_F : FiniteField
        Asal alt cisim GF(2).
    ext_degree : int
        F_q'nun GF(2) üzerindeki genişleme derecesi (k).
    R : MPolynomialRing
        n değişkenli, F_q katsayılı çok değişkenli polinom halkası.
    vars : vector
        R halkasının üreteçlerinden oluşan vektör.
    T_mat : Matrix
        Gizli tersinir n x n doğrusal dönüşüm matrisi.
    P_matrices : list of Matrix
        Açık anahtarın her bir bileşenine ait ÜST ÜÇGENSEL kuadratik
        form matrisi (karakteristik 2'de kuadratik formun doğal temsili
        budur; simetrik hali ayrıca Q_bar_matrices'te tutulur).
    Q_bar_matrices : list of Matrix
        Her P_k için hesaplanan simetrik bilineer form matrisi
        (Q_bar_k = P_k + P_k^T); saldırının W1, W2 doğrusal
        kombinasyonları bu matrisler üzerinden kurulur.
    P_polys : list of MPolynomial
        Açık anahtarı oluşturan o adet homojen kuadratik polinom.
    Secret_Oil_Space : VectorSpace
        Gizli Oil uzayının açık koordinat sistemine taşınmış hali;
        YALNIZCA simülasyon doğrulaması amacıyla saklanır.
    """
    def __init__(self, q, o):
        if q % 2 != 0:
            raise ValueError("Bu kod SADECE çift karakteristik (q=2^k) içindir!")

        self.q = q
        self.o = o
        # Kipnis-Shamir saldırısının bu öğretici sürümünde dengeli OV durumu ele alınır.
        # Bu nedenle v = o alınmıştır.
        self.v = o
        self.n = self.o + self.v

        # F_q = GF(2^k) temel sonlu cismi ve asal alt cismi GF(2); cisim
        # inişi (field descent) tekniği, F_q elemanlarını base_F
        # üzerindeki koordinat vektörlerine (ext_degree boyutlu)
        # çevirmeye dayanır.
        self.F = GF(q, 'a')
        self.base_F = GF(2)
        self.ext_degree = self.F.degree()

        self.R = PolynomialRing(self.F, self.n, 'x')
        self.vars = vector(self.R, self.R.gens())

        print("\n" + "="*80)
        print(f"KIPNIS-SHAMIR SALDIRISI (DETAYLI İZLEME MODU)")
        print(f"Parametreler: GF({q}), o={o}, v={self.v}, n={self.n}")
        print("="*80)

        # Gizli ve açık anahtar bileşenleri generate_keys() çağrılana
        # kadar boş/None tutulur; Secret_Oil_Space yalnızca simülasyon
        # doğrulaması amaçlı bir "yan kanal" bilgisidir.
        self.T_mat = None
        self.P_matrices = []
        self.Q_bar_matrices = []
        self.P_polys = []
        self.Secret_Oil_Space = None

    def print_visual_matrix(self, M, name="Matris"):
        """
        Bir matrisi, Oil x Oil bloğundaki köşegen ve köşegen dışı
        girdileri özel işaretlerle vurgulayarak biçimlendirilmiş
        biçimde ekrana yazdırır.

        Bu görselleştirme, denk anahtar matrislerinin (F') beklenen
        blok yapısını (Oil x Oil köşegen dışı girdilerin SIFIR olması,
        köşegen girdilerin ise serbest kalabilmesi) doğrudan gözle
        denetleyebilmek için kullanılır: '*' işareti Oil x Oil
        köşegenindeki sıfır olmayan bir değeri, '!' işareti ise Oil x
        Oil köşegen DIŞI (beklenmedik biçimde sıfır olmayan) bir
        değeri belirtir.

        Parametreler
        ------------
        M : Matrix
            Görselleştirilecek n x n matris (tipik olarak bir F'_k
            denk merkezi dönüşüm matrisi).
        name : str, optional
            Çıktıda matrisin başlığında kullanılacak isim (varsayılan:
            "Matris").

        Dönüş Değeri
        ------------
        None
            Yalnızca ekrana biçimlendirilmiş çıktı yazdırır.
        """
        rows = M.nrows()
        cols = M.ncols()
        print(f"\n   --- {name} (Görsel Analiz) ---")
        print("   " + "-" * (cols * 9 + 4))
        for r in range(rows):
            row_str = "   | "
            for c in range(cols):
                val = M[r,c]
                is_oil = (r >= self.v and c >= self.v)
                is_diag = (r == c)
                val_str = f"*{val}*" if (is_oil and is_diag and val!=0) else (f"!{val}!" if (is_oil and val!=0) else f"{val}")
                row_str += f"{val_str:^7} "
                if c == self.v - 1: row_str += "| "
            print(row_str + "|")
            if r == self.v - 1:
                print("   |" + "-" * (self.v * 8 + 1) + "+" + "-" * (self.o * 8 + 1) + "|")
        print("   " + "-" * (cols * 9 + 4) + "\n")

    def _to_base_vector(self, elem):
        """
        GF(2^m) elemanını GF(2) üzerindeki koordinat vektörüne dönüştürür.
        Örneğin GF(4)=GF(2)[a] için a+1 -> (1,1).

        Teorik Gerekçe
        ---------------
        Bu metot, "cisim inişi" (field descent / Weil descent)
        tekniğinin temel yapı taşıdır. GF(2^k), GF(2) üzerinde
        ext_degree = k boyutlu bir vektör uzayı olarak görülebilir; bu
        izomorfizma sayesinde GF(2^k) üzerindeki DOĞRUSAL (veya
        Frobenius/kare alma içeren) denklemler, GF(2) üzerinde tanımlı
        daha büyük boyutlu fakat SAF DOĞRUSAL bir denklem sistemine
        indirgenebilir. run_attack() içindeki imza sahtekarlığı
        aşamasında, GF(2^k) üzerindeki her bir kuadratik/doğrusal
        denklem, bu metot aracılığıyla GF(2) üzerinde ext_degree adet
        doğrusal denkleme "parçalanır".

        Parametreler
        ------------
        elem : FiniteFieldElement
            GF(2^k) cisminden bir eleman.

        Dönüş Değeri
        ------------
        vector
            Uzunluğu ext_degree olan, base_F = GF(2) üzerinde tanımlı
            koordinat vektörü.
        """
        coords = list(vector(elem))

        # Güvenlik için uzunluğu ext_degree'e tamamlıyoruz.
        while len(coords) < self.ext_degree:
            coords.append(self.base_F(0))

        return vector(self.base_F, coords[:self.ext_degree])

    def generate_keys(self):
        """
        Çift karakteristikli (q = 2^k) Balanced OV şemasının gizli
        anahtarını (T dönüşümü, gizli Oil uzayı ve merkezi dönüşüm F)
        rastgele üretir ve bunlardan açık anahtarı (P_matrices,
        Q_bar_matrices, P_polys) türetir.

        Algoritmanın Adımları
        -----------------------
        1. Gizli T Dönüşümü:
           n x n boyutunda TERSİNİR rastgele bir matris T_mat seçilir;
           bu, merkezi dönüşümün Oil/Vinegar blok yapısını saldırgandan
           gizleyen maskeleme katmanıdır.

        2. Gizli Oil Uzayı:
           Merkezi koordinat sisteminde Oil uzayı, standart baz
           vektörlerinin son o tanesi tarafından gerilir; T^{-1} ile
           açık koordinat sistemine taşınarak Secret_Oil_Space elde
           edilir (yalnızca simülasyon doğrulaması için saklanır).

        3. Merkezi Dönüşüm (F) ve Açık Anahtar (P):
           Her k için, ÜST ÜÇGENSEL bir matris M_k rastgele üretilir;
           bu matriste sağ alt o x o (Oil x Oil) bloğunun KÖŞEGEN DIŞI
           girdileri sıfır bırakılır (yalnızca Oil x Oil köşegen
           girdileri, yani x_i^2 terimlerinin katsayıları, serbestçe
           rastgele seçilebilir -- bu, karakteristik 2'de "Oil x Oil
           çarpraz terim yok" kısıtlamasının doğru yorumudur, çünkü
           x_i^2 karakteristik 2'de doğrusal bir Frobenius görüntüsü
           gibi davranır ve OV'nin trapdoor mekanizmasını bozmaz).
           Merkezi polinom poly_f = x^T * M_k * x hesaplanıp T
           dönüşümü sembolik olarak yerine konularak açık anahtar
           polinomu poly_p = poly_f(T(x)) elde edilir. Bu polinomdan,
           köşegen ve köşegen üstü katsayılar okunarak ÜST ÜÇGENSEL
           açık anahtar matrisi P_k yeniden inşa edilir (bu matris
           artık MERKEZİ M_k'nın blok yapısını YANSITMAZ). Son olarak,
           saldırının W1, W2 kombinasyonlarında kullanacağı SİMETRİK
           bilineer form matrisi Q_bar_k = P_k + P_k^T hesaplanır
           (karakteristik 2'de bu toplam, köşegen üzerinde 2*P_k[i,i]
           = 0 verdiğinden köşegeni SIFIRLAR ve yalnızca köşegen dışı
           çift-yönlü katkıları taşır -- bu, karakteristik 2'ye özgü
           önemli bir cebirsel inceliktir).

        Yan Etkiler
        -----------
        self.T_mat, self.Secret_Oil_Space, self.P_matrices,
        self.Q_bar_matrices ve self.P_polys öznitelikleri bu metot
        tarafından (yeniden) doldurulur. Metot birden fazla kez
        çağrılırsa önceki anahtar bileşenleri temizlenip yeniden
        üretilir.

        Dönüş Değeri
        ------------
        None
        """
        # generate_keys() birden fazla kez çağrılırsa eski veriler temizlenir.
        self.P_matrices = []
        self.Q_bar_matrices = []
        self.P_polys = []
        self.Secret_Oil_Space = None

        print("\n" + "*"*30 + " ADIM 1: SİSTEM İNŞASI " + "*"*30)

        # 1. T Matrisi
        # T_mat, gizli merkezi dönüşümün Oil/Vinegar blok yapısını
        # saldırgandan gizleyen tersinir bir doğrusal dönüşümdür.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible(): break
        print(f"\n[1.1] Gizli T Matrisi:\n{self.T_mat}")

        # 2. Gizli Oil Uzayı
        # Merkezi (gizli) koordinat sisteminde Oil uzayı, standart baz
        # vektörlerinin son o tanesi tarafından gerilir; T^{-1} ile
        # açık koordinat sistemine taşınarak saldırının nihai hedefi
        # olan Secret_Oil_Space elde edilir.
        std_oil_basis = []
        for i in range(self.v, self.n):
            vec = vector(self.F, [0]*self.n)
            vec[i] = 1
            std_oil_basis.append(vec)
        T_inv = self.T_mat.inverse()
        self.Secret_Oil_Space = VectorSpace(self.F, self.n).span([T_inv * v for v in std_oil_basis])
        print(f"\n[1.2] Gizli Oil Uzayı Bazı:\n{self.Secret_Oil_Space.basis_matrix()}")

        # 3. P Matrisleri
        print(f"\n[1.3] Açık Anahtar Üretimi...")
        for k in range(self.o):
            # M_k: üst üçgensel merkezi matris. Sağ alt o x o bloğunda
            # yalnızca KÖŞEGEN girdiler (x_i^2 katsayıları) rastgele
            # seçilir; köşegen dışı Oil x Oil girdiler (x_i*x_j,
            # i != j) sıfır bırakılır. Bu, karakteristik 2'de OV'nin
            # trapdoor kısıtlamasının doğru biçimidir.
            M_k = matrix(self.F, self.n, self.n)
            for i in range(self.n):
                for j in range(i, self.n):
                    if i >= self.v and j >= self.v:
                        M_k[i, j] = self.F.random_element() if i==j else 0
                    else:
                        M_k[i, j] = self.F.random_element()

            poly_f = self.vars * M_k * self.vars
            # T dönüşümü sembolik olarak yerine konularak açık anahtar
            # polinomu elde edilir: poly_p(x) = poly_f(T(x)).
            poly_p = poly_f.subs({self.vars[i]: (self.T_mat * self.vars)[i] for i in range(self.n)})
            self.P_polys.append(poly_p)

            # Açık anahtar polinomundan, köşegen (x_i^2) ve köşegen üstü
            # (x_i*x_j) katsayılar okunarak ÜST ÜÇGENSEL P_k matrisi
            # yeniden inşa edilir. Bu adım, karakteristik 2'de kuadratik
            # formun matris temsilinin polinomdan NASIL geri okunması
            # gerektiğini gösterir.
            P_k = matrix(self.F, self.n, self.n)
            for i in range(self.n):
                P_k[i,i] = poly_p.monomial_coefficient(self.vars[i]**2)
                for j in range(i+1, self.n):
                    P_k[i,j] = poly_p.monomial_coefficient(self.vars[i]*self.vars[j])

            self.P_matrices.append(P_k)
            # Q_bar_k = P_k + P_k^T: karakteristik 2'de köşegen üzerinde
            # 2*P_k[i,i] = 0 verdiğinden, bu simetrik matris köşegeni
            # sıfırlar ve saldırının W1, W2 kombinasyonlarında
            # kullanacağı "temiz" simetrik bilineer formu temsil eder.
            self.Q_bar_matrices.append(P_k + P_k.transpose())
            # --- [DEĞİŞİKLİK 1] DETAYLI ANAHTAR ÇIKTISI ---
            print(f"\n   >>> Bileşen k={k} <<<")
            print(f"   Gizli F_{k} Matrisi (Sağ Alt Blok Köşegeni Dolu Olabilir):\n{M_k}")
            print(f"   Açık P_{k} Matrisi (Üst Üçgensel):\n{P_k}")
            print(f"   Açık Polinom p_{k}(x):\n   {poly_p}")
            # -----------------------------------------------
            # --- [EKLEME] Q MATRİSLERİNİ GÖRÜNTÜLEME ---
            Q_k = self.Q_bar_matrices[-1] # Listeye en son eklenen Q matrisini al
            print(f"   Simetrik Q_{k} Matrisi (P_{k} + P_{k}^T):\n{Q_k}")
            # -------------------------------------------

    def _attempt_subspace_recovery(self, attempt_num):
        """
        Açık anahtarın simetrik bilineer form matrislerinden (Q_bar_k)
        rastgele bir doğrusal kombinasyon çifti (W1, W2) oluşturarak,
        karakteristik-2'ye özgü teknikle gizli Oil uzayının bir adayını
        yeniden inşa etmeye çalışan TEK BİR denemeyi gerçekleştirir.

        Algoritmanın Adımları
        -----------------------
        1. W1, W2 Oluşturma:
           Q_bar_matrices'in rastgele katsayılı doğrusal
           kombinasyonlarından W1 ve W2 matrisleri kurulur. W1'in
           tersinir olması zorunludur; aksi halde deneme başarısız
           sayılır (None döner).

        2. Matris Demeti (W1, W2 Çifti) Analizi:
           W = W1^{-1} * W2 hesaplanır. Balanced OV'nin merkezi
           dönüşümündeki blok yapısı nedeniyle W'nin karakteristik
           polinomu C(lambda), karakteristik 2'de C(lambda) =
           C1(lambda)^2 biçiminde TAM KARE olmalıdır.

        3. Karakteristik 2'de Polinom Kareköpü:
           "Freshman's Dream" özdeşliği (a+b)^2 = a^2+b^2 sayesinde,
           C(lambda)'nın TEK indisli katsayılarının sıfır olması
           denetlenir (tam karelik ön koşulu); sağlanıyorsa, ÇİFT
           indisli katsayıların (GF(2^k)'da benzersiz) kareköpü
           alınarak C1(lambda) doğrudan inşa edilir.

        4. Katlılığı 1 Olan Kökün Seçimi:
           C1(lambda)'nın kökleri hesaplanır. Kaynak algoritmaya göre
           (bkz. tez Algoritma 5.2), katlılığı (multiplicity) TAM 1
           olan bir DOĞRUSAL kök, saldırı için en güvenilir adaydır;
           böyle bir kök bulunursa lambda_val olarak seçilir. Hiçbir
           kök bu koşulu sağlamazsa deneme başarısız sayılır.

        5. Özuzaydan Tohum Vektörlerin Seçimi:
           Karakteristik 2'de W - lambda*I = W + lambda*I olduğundan,
           target = W + lambda_val * I matrisinin sağ çekirdeği
           (özuzay) hesaplanır. Bu özuzaydan iki bağımsız "tohum"
           vektör (v1, v2) alınır; bu vektörler ve bunların F_q
           katsayılı doğrusal kombinasyonları (v1 + k*v2, k in F_q)
           birer Oil uzayı "başlangıç noktası" adayı oluşturur.

        6. Adayların Genişletilmesi:
           Her aday tohum vektör için, rastgele yeni W1_tmp, W2_tmp
           kombinasyonları üretilip elde edilen W_rand matrisi mevcut
           aday uzayın bazına uygulanarak uzay adım adım
           GENİŞLETİLİR (bu, gizli Oil uzayının W_rand altında
           DEĞİŞMEZ olmasından yararlanan bir yineleme stratejisidir).
           Genişleme, hedef boyuta (o) ulaşılana veya deneme sınırına
           (2*o adım) varılana kadar sürer.

        7. Simülasyon Doğrulaması:
           Genişletilen adayın gizli Secret_Oil_Space'in bir alt uzayı
           olup olmadığı kontrol edilir. Bu kontrol YALNIZCA öğretici
           simülasyonun doğruluğunu teyit etmek içindir; gerçek bir
           saldırgan bu bilgiye erişemez.

        Parametreler
        ------------
        attempt_num : int
            Bu deneme çağrısının sıra numarası; yalnızca ekran
            çıktısında ilerlemeyi izlemek amacıyla kullanılır.

        Dönüş Değeri
        ------------
        VectorSpace veya None
            Başarılı olursa, boyutu o olan ve Secret_Oil_Space'in bir
            alt uzayı olduğu doğrulanmış aday Oil uzayı; herhangi bir
            adımda başarısız olunursa None.
        """
        print(f"\n   --- Alt Uzay Kurtarma Denemesi #{attempt_num} ---")

        w1_coeffs = [self.F.random_element() for _ in range(self.o)]
        w2_coeffs = [self.F.random_element() for _ in range(self.o)]
        print(f"   Rastgele Katsayılar (W1): {w1_coeffs}")
        print(f"   Rastgele Katsayılar (W2): {w2_coeffs}")

        # --- [DEĞİŞİKLİK 2] MATRİS DEMETİ (W1, W2 ÇİFTİ) ADIMLARI ---
        # W1 ve W2, açık anahtarın simetrik bilineer form matrislerinin
        # (Q_bar_matrices) rastgele doğrusal kombinasyonlarıdır. W1'in
        # tersinir olması, W = W1^{-1} * W2 matris demetinin
        # tanımlanabilmesi için zorunludur.
        W1 = matrix(self.F, self.n, self.n)
        W2 = matrix(self.F, self.n, self.n)

        print(f"\n   [Açık Anahtar Matrislerinden Rastgele Lineer Kombinasyonlar Oluşturuluyor]")
        print(f"   Rastgele Katsayılar (alfa): {w1_coeffs}")
        print(f"   Rastgele Katsayılar (beta): {w2_coeffs}")

        for k in range(self.o):
            W1 += w1_coeffs[k] * self.Q_bar_matrices[k]
            W2 += w2_coeffs[k] * self.Q_bar_matrices[k]
            print(f"      Adım {k}: W1 += {w1_coeffs[k]} * Q_{k}, W2 += {w2_coeffs[k]} * Q_{k}")

        print(f"   -> Oluşan W1 Matrisi:\n{W1}")
        print(f"   -> Oluşan W2 Matrisi:\n{W2}")
        # -----------------------------------------------

        if not W1.is_invertible():
            print("   [HATA] W1 matrisi tersinir değil.")
            return None

        W = W1.inverse() * W2
        print(f"   W Matrisi:\n{W}")

        char_poly = W.characteristic_polynomial()
        print(f"   Karakteristik Polinom: {char_poly}")

        try:
            coeffs = char_poly.coefficients(sparse=False)

            # Karakteristik 2'de tam kare polinomda tek dereceli katsayılar sıfır olmalıdır.
            for i in range(1, len(coeffs), 2):
                if coeffs[i] != 0:
                    print("   [UYARI] Karakteristik polinom tam kare formunda değil. Yeni W1,W2 denenmeli.")
                    return None

            # C(lambda)=C1(lambda)^2 olduğunda çift dereceli katsayıların karekökleri alınır.
            c1_coeffs = [coeffs[i].sqrt() for i in range(0, len(coeffs), 2)]
            C1 = char_poly.parent()(c1_coeffs)

            print(f"   Karekök Polinom C1(x): {C1}")

        except Exception as e:
            print(f"   [HATA] Karekök alma başarısız: {e}")
            return None

        roots = C1.roots()
        # =================== KÖK ANALİZİ BAŞLANGICI ===================
        print(f"   Ham Kök Çıktısı: {roots}")
        print(f"   [BİLGİ] SageMath Formatı: [(Kök Değeri, Tekrar Sayısı), ...]")

        target_root = None

        for r_idx, (r_val, r_mult) in enumerate(roots):
            print(f"      -> Kök #{r_idx+1}: Değer (Lambda) = {r_val}")
            print(f"         Katlılık (Multiplicity) = {r_mult}")

            # Yorum Satırı: Kaynak kitap (Algoritma 5.2) ne diyor?
            # "Look for a distinctive linear factor of multiplicity 1."
            if r_mult == 1:
                print("         [*** MÜKEMMEL ***] Bu kökün katlılığı 1. Saldırı için en ideal aday!")
                if target_root is None: target_root = r_val # İlk bulduğumuz ideal kökü seçelim
            else:
                print("         [---] Katlılık 1 değil. (Yine de denenebilir ama ideal değil)")

        # Eğer hiç ideal (katlılığı 1 olan) kök yoksa, mecburen ilkini alacağız
        if target_root is None:
            print("   [UYARI] Katlılığı 1 olan lineer kök bulunamadı. Yeni W1,W2 denenmelidir.")
            return None

        # Lambda değerini güncelleyelim (Otomatik seçim)
        lambda_val = target_root
        print(f"   => Seçilen Lambda: {lambda_val}")
        # =================== KÖK ANALİZİ BİTİŞİ =======================
        print(f"   Bulunan Kökler: {roots}")
        if not roots:
            return None

        # Karakteristik 2 olduğundan W - lambda*I = W + lambda*I olur.
        # Burada lambda_val yukarıda seçilen target_root değeridir.
        target = W + lambda_val * matrix.identity(self.F, self.n)
        eigenspace = target.right_kernel()
        basis = eigenspace.basis()
        print(f"   Özuzay Boyutu: {len(basis)}")

        if len(basis) < 2: return None

        v1, v2 = basis[0], basis[1]
        # --- [EKLEME] BAZ VEKTÖRLERİNİ GÖRÜNTÜLEME ---
        print(f"   [DETAY] Özuzay Baz Vektörleri (Tohumlar):")
        print(f"      v1: {v1}")
        print(f"      v2: {v2}")
        # ---------------------------------------------
        candidates = [v2] + [v1 + k*v2 for k in self.F]
        print(f"   Oluşturulan Aday Sayısı: {len(candidates)}")

        for idx, s_vec in enumerate(candidates):
            current_space = VectorSpace(self.F, self.n).subspace([s_vec])
            steps = 0
            print(f"      Aday {idx} genişletiliyor... Başlangıç Boyutu: {current_space.dimension()}")

            while current_space.dimension() < self.o and steps < 2 * self.o:
                w1_c = [self.F.random_element() for _ in range(self.o)]
                w2_c = [self.F.random_element() for _ in range(self.o)]
                W1_tmp = sum(c * M for c, M in zip(w1_c, self.Q_bar_matrices))
                W2_tmp = sum(c * M for c, M in zip(w2_c, self.Q_bar_matrices))

                if W1_tmp.is_invertible():
                    W_rand = W1_tmp.inverse() * W2_tmp
                    new_vecs = [W_rand * b for b in current_space.basis()]
                    current_space += VectorSpace(self.F, self.n).subspace(new_vecs)
                steps += 1

            print(f"      -> Sonuç Boyutu: {current_space.dimension()} ({steps} adımda)")

            if current_space.is_subspace(self.Secret_Oil_Space):
                print("      -> [SİMÜLASYON KONTROLÜ] Bulunan uzay gizli oil uzayıyla eşleşiyor.")
                return current_space
            else:
                print("      -> [SİMÜLASYON KONTROLÜ] Boyut doğru ama gizli oil uzayıyla eşleşmedi.")
        return None

    def run_attack(self):
        """
        Açık anahtar üzerinden çift karakteristikli Kipnis-Shamir
        saldırısını uçtan uca yürütür: gizli Oil uzayının (retry
        mekanizmasıyla) yeniden inşası, denk anahtar çiftinin (T', F')
        oluşturulması ve cisim inişi tekniğiyle sahte imza üretimi.

        Algoritmanın Adımları
        -----------------------
        1. Alt Uzay Kurtarma (Retry):
           _attempt_subspace_recovery en fazla 20 kez çağrılır; her
           çağrı farklı rastgele W1, W2 kombinasyonları dener. İlk
           başarılı deneme found_space olarak kabul edilir; hiçbiri
           başarılı olmazsa saldırı durdurulur.

        2. Denk Anahtar (T', F') İnşası:
           Bulunan Oil uzayının bazı ile bu uzayın bölüm uzayından
           (quotient space) elde edilen Vinegar tamamlayıcı baz
           birleştirilerek saldırganın T'^{-1} matrisi kurulur.
           Her açık anahtar matrisi P_k için,
                F'_k = T'^{-1,T} * P_k * T'^{-1}
           hesaplanarak denk merkezi dönüşüm matrisleri elde edilir; bu
           matrislerin görsel analizi (print_visual_matrix) ve kuadratik
           form polinomu (poly_prime) ekrana yazdırılır.

        3. Cisim İnişiyle Sahte İmza (Forgery):
           Rastgele bir mesaj (msg) ve rastgele Vinegar değerleri
           (v_val) seçilir. Her F'_k matrisinden Vinegar-Vinegar
           (F_vv), Vinegar-Oil çapraz (F_vo, F_ov) ve Oil-Oil köşegen
           (F_oo) blokları çıkarılarak Oil değişkenlerine göre
           KUADRATİK bir denklem (doğrusal terim + x_i^2 terimi)
           kurulur. Bu GF(2^k) üzerindeki denklem, _to_base_vector
           yardımıyla GF(2) üzerinde tanımlı ext_degree adet DOĞRUSAL
           denkleme "indirgenir" (cisim inişi / field descent); Oil
           değişkeni sayısı kadar tekrarlanan bu işlem sonunda GF(2)
           üzerinde büyük bir doğrusal sistem (S_Mat, S_Vec) elde
           edilir.

        4. Çözümün İnşası ve Doğrulama:
           GF(2) sistemi çözülüp elde edilen bitler, F_q'nun bir bazı
           (basis_F, Frobenius kuvvetleri) kullanılarak GF(2^k)
           elemanlarına geri dönüştürülür (Oil değerleri). Vinegar ve
           Oil değerleri birleştirilip T'^{-1} ile gerçek imzaya
           dönüştürülür; imza, ORİJİNAL açık anahtar polinomlarında
           (P_polys) değerlendirilerek doğrulanır.

        Ön Koşullar
        ------------
        generate_keys() metodunun bu metottan önce çağrılmış olması
        gerekir (aksi halde P_matrices/P_polys boş kalacağından ilgili
        adımlar anlamsız sonuçlar üretir).

        Dönüş Değeri
        ------------
        None
            Tüm ara sonuçlar (bulunan Oil uzayı, denk anahtar
            matrisleri, sahte imza ve doğrulama sonucu) ayrıntılı
            biçimde ekrana yazdırılır; bir değer döndürülmez. Alt uzay
            kurtarma başarısız olursa veya GF(2) doğrusal sistemi
            çözümsüzse fonksiyon erken sonlanır.
        """
        print("\n" + "*"*30 + " ADIM 2: SALDIRI (RETRY) " + "*"*30)
        found_space = None
        for attempt in range(1, 20):
            found_space = self._attempt_subspace_recovery(attempt)
            if found_space: break

        if not found_space:
            print("\n[BAŞARISIZ] Uzay bulunamadı.")
            return

        print(f"\n[BAŞARI] Oil Uzayı Bazı:\n{found_space.basis_matrix()}")

        print("\n" + "*"*30 + " ADIM 3: DENK ANAHTAR (T', F') " + "*"*30)
        # Bulunan Oil uzayının bazı, bölüm uzayından (quotient space)
        # elde edilen bir Vinegar tamamlayıcı baz ile birleştirilerek
        # saldırganın kendi T'^{-1} matrisi (Vinegar || Oil sütun
        # sıralamasıyla) kurulur.
        oil_basis = found_space.basis()
        vinegar_basis = VectorSpace(self.F, self.n).quotient(found_space).lift_map().image().basis()
        T_prime_inv = matrix(self.F, list(vinegar_basis) + list(oil_basis)).transpose()
        print(f"Denk T'^(-1) Matrisi:\n{T_prime_inv}")

        F_prime_matrices = []
        for k in range(self.o):
            # F'_k = T'^{-1,T} * P_k * T'^{-1}: açık anahtar matrisi,
            # saldırganın kendi (varsayımsal) Vinegar/Oil koordinat
            # sistemine geri taşınır; bu matrisin Oil x Oil köşegen
            # dışı bloğunun sıfır çıkması, bulunan Oil uzayının
            # doğruluğunun somut kanıtıdır.
            F_prime = T_prime_inv.transpose() * self.P_matrices[k] * T_prime_inv
            F_prime_matrices.append(F_prime)
            self.print_visual_matrix(F_prime, name=f"F'_{k}")
            # --- [DEĞİŞİKLİK 3] DENK MATRİS DEĞERLERİ ---
            print(f"   F'_{k} Matrisinin Sayısal Hali:\n{F_prime}")
            # ---------------------------------------------
            # --- [EKLEME] DENK POLİNOMU YAZDIRMA ---
            # Matris formundan (x^T * F * x) polinom formuna geçiş
            poly_prime = self.vars * F_prime * self.vars
            print(f"   Denk Polinom f'_{k}(x) [Kuadratik Form]:\n   {poly_prime}")
            print("-" * 60)
            # ---------------------------------------

        print("\n" + "*"*30 + " ADIM 4: İMZA SAHTECİLİĞİ  " + "*"*30)
        msg = random_vector(self.F, self.o)
        v_val = random_vector(self.F, self.v)
        # basis_F: F_q'nun GF(2) üzerindeki tabanı, Frobenius otomorfizmasının
        # (x -> x^2) kuvvetleri (self.F.gen()**i) olarak seçilmiştir; cisim
        # inişi sırasında GF(2) katsayılarını geri GF(2^k)'ya taşımak için
        # kullanılır.
        basis_F = [self.F.gen()**i for i in range(self.ext_degree)]

        print(f"Hedef Mesaj: {msg}")
        print(f"Vinegar: {v_val}")

        linear_eqs = []
        linear_rhss = []

        for k in range(self.o):
            F_mat = F_prime_matrices[k]
            F_vv = F_mat.submatrix(0, 0, self.v, self.v)
            const_val = v_val * F_vv * v_val

            F_vo = F_mat.submatrix(0, self.v, self.v, self.o)
            F_ov = F_mat.submatrix(self.v, 0, self.o, self.v)
            lin_coeff_vec = (v_val * F_vo) + (F_ov * v_val)

            F_oo = F_mat.submatrix(self.v, self.v, self.o, self.o)
            quad_coeff_vec = [F_oo[i,i] for i in range(self.o)]

            # Karakteristik 2 olduğundan çıkarma ile toplama aynıdır.
            rhs = msg[k] + const_val

            # Print Equation
            eq_str = []
            for i in range(self.o):
                if quad_coeff_vec[i] != 0: eq_str.append(f"({quad_coeff_vec[i]})*o{i}^2")
                if lin_coeff_vec[i] != 0: eq_str.append(f"({lin_coeff_vec[i]})*o{i}")
            print(f"   Denklem {k}: {' + '.join(eq_str)} = {rhs}")

            # Field Descent Logic
            # Bu döngü, GF(2^k) üzerindeki tek bir kuadratik denklemi,
            # GF(2) üzerinde tanımlı ext_degree adet DOĞRUSAL denkleme
            # "parçalar": her bir GF(2^k) katsayısı (doğrusal veya
            # kuadratik), F_q'nun tabanındaki her bir eleman (basis_F)
            # ile çarpılıp _to_base_vector aracılığıyla GF(2)
            # koordinatlarına indirgenir.
            for dim in range(self.ext_degree):
                row = [self.base_F(0)] * (self.o * self.ext_degree)

                rhs_vec = self._to_base_vector(rhs)
                rhs_comp = rhs_vec[dim]

                for oil_idx in range(self.o):
                    l_c = lin_coeff_vec[oil_idx]

                    for bit in range(self.ext_degree):
                        vec_l = self._to_base_vector(l_c * basis_F[bit])
                        row[oil_idx*self.ext_degree + bit] += vec_l[dim]

                    q_c = quad_coeff_vec[oil_idx]

                    for bit in range(self.ext_degree):
                        vec_q = self._to_base_vector(q_c * (basis_F[bit]**2))
                        row[oil_idx*self.ext_degree + bit] += vec_q[dim]

                linear_eqs.append(row)
                linear_rhss.append(rhs_comp)

        S_Mat = matrix(self.base_F, linear_eqs)
        S_Vec = vector(self.base_F, linear_rhss)

        print(f"\n   --- İndirgenmiş GF(2) Matrisi ({S_Mat.nrows()}x{S_Mat.ncols()}) ---")
        print(f"{S_Mat}")
        print(f"   Sağ Taraf (b): {S_Vec}")

        try:
            sol_bits = S_Mat.solve_right(S_Vec)

            # =================== EKLEME BAŞLANGICI ===================
            print(f"\n   [ADIM 4.2] Çözümün İnşası (Reconstruction)")
            print(f"      Field Descent Çözüm Bitleri: {sol_bits}")
            print(f"      Baz Elemanları: {basis_F}")

            oil_vals = []

            for i in range(self.o):
                bits = sol_bits[i*self.ext_degree : (i+1)*self.ext_degree]

                val = self.F(0)
                for j in range(self.ext_degree):
                    val += self.F(bits[j]) * basis_F[j]

                oil_vals.append(val)

                print(f"      -> Oil_{i} Bitleri {bits} => Değer: {val}")

            oil_vec = vector(self.F, oil_vals)
            print(f"   -> Bulunan Oil Vektörü (o'): {oil_vec}")

            # 3. Denk Çözüm Vektörü (z')
            z_prime = vector(self.F, list(v_val) + list(oil_vals))
            print(f"\n   [ADIM 4.3] Denk Uzaydaki Çözüm Vektörü (z' = v' || o')")
            print(f"      z': {z_prime}")

            # 4. Gerçek İmzaya Dönüşüm
            print(f"\n   [ADIM 4.4] Gerçek İmzaya Dönüşüm (x = T'^(-1) * z')")
            print(f"      Dönüşüm: T'^(-1) matrisi ile z' çarpılıyor...")

            signature = T_prime_inv * z_prime
            print(f"      SAHTE İMZA (x): {signature}")
            # =================== EKLEME BİTİŞİ =======================

            # Verify (Mevcut Doğrulama Kodun Buradan Devam Eder)
            print("\n" + "*"*30 + " ADIM 5: DOĞRULAMA " + "*"*30)
            res = []
            for p in self.P_polys:
                sub = {self.vars[i]: signature[i] for i in range(self.n)}
                res.append(p.subs(sub))

            res_vec = vector(self.F, res)
            print(f"   P(imza) Sonucu : {res_vec}")
            print(f"   Hedef Mesaj    : {msg}")

            if res_vec == msg:
                print("   >>> SONUÇ: BAŞARILI! Sistem tamamen kırıldı. <<<")
            else:
                print("   >>> SONUÇ: HATA (Doğrulama başarısız).")

        except ValueError:
            print("\n   [HATA] Çözüm yok (Bu Vinegar değerleri için sistem tutarsız).")

# ============================================================================
# 1. SİSTEM KURULUMU
# ============================================================================
# GF(4) üzerinde dengeli OV sistemi oluşturulur.
# Bu hücre çalıştırıldığında yeni gizli anahtar, açık anahtar ve gizli oil uzayı üretilir.
# Bu blok, KipnisShamirEvenCharacteristicDemo sınıfının tam bir yaşam
# döngüsünü (anahtar üretimi -> saldırı) örneklendiren bir DEMO/TEST
# senaryosudur. Kütüphane olarak içe aktarılırken çalışmasını
# engellemek isteyen kullanıcılar bu bloğu `if __name__ == "__main__":`
# koruması altına alabilir.

my_q = 4
my_o = 3

ks = KipnisShamirEvenCharacteristicDemo(q=my_q, o=my_o)
ks.generate_keys()




KIPNIS-SHAMIR SALDIRISI (DETAYLI İZLEME MODU)
Parametreler: GF(4), o=3, v=3, n=6

****************************** ADIM 1: SİSTEM İNŞASI ******************************

[1.1] Gizli T Matrisi:
[    1     1     1     0 a + 1     0]
[    0     a     1     a     1     a]
[a + 1     0     a     0     0     0]
[    a     0     1     0     1     a]
[    0     a     0     a     0 a + 1]
[    1     a a + 1     0 a + 1     0]

[1.2] Gizli Oil Uzayı Bazı:
[1 0 a 0 1 a]
[0 1 0 0 a 0]
[0 0 0 1 0 1]

[1.3] Açık Anahtar Üretimi...

   >>> Bileşen k=0 <<<
   Gizli F_0 Matrisi (Sağ Alt Blok Köşegeni Dolu Olabilir):
[    a     0 a + 1 a + 1 a + 1     1]
[    0     1     0     0     1 a + 1]
[    0     0     1     a     a     a]
[    0     0     0 a + 1     0     0]
[    0     0     0     0     1     0]
[    0     0     0     0     0 a + 1]
   Açık P_0 Matrisi (Üst Üçgensel):
[    0     0     0     a     0 a + 1]
[    0     a     a a + 1     a a + 1]
[    0     0     0     1     a     0]
[    0     0     

In [15]:
# ============================================================================
# AYNI ÖRNEK ÜZERİNDE SALDIRI DENEMESİ
# ============================================================================
# Bu hücreyi tekrar çalıştırmak yeni anahtar üretmez.
# Sadece aynı açık anahtar üzerinde farklı rastgele W1, W2 seçimleriyle
# saldırıyı tekrar dener.

ks.run_attack()



****************************** ADIM 2: SALDIRI (RETRY) ******************************

   --- Alt Uzay Kurtarma Denemesi #1 ---
   Rastgele Katsayılar (W1): [a, 0, a + 1]
   Rastgele Katsayılar (W2): [0, 1, a]

   [Açık Anahtar Matrislerinden Rastgele Lineer Kombinasyonlar Oluşturuluyor]
   Rastgele Katsayılar (alfa): [a, 0, a + 1]
   Rastgele Katsayılar (beta): [0, 1, a]
      Adım 0: W1 += a * Q_0, W2 += 0 * Q_0
      Adım 1: W1 += 0 * Q_1, W2 += 1 * Q_1
      Adım 2: W1 += a + 1 * Q_2, W2 += a * Q_2
   -> Oluşan W1 Matrisi:
[    0     0     0     1 a + 1 a + 1]
[    0     0     0 a + 1     0 a + 1]
[    0     0     0     1 a + 1     0]
[    1 a + 1     1     0 a + 1     0]
[a + 1     0 a + 1 a + 1     0 a + 1]
[a + 1 a + 1     0     0 a + 1     0]
   -> Oluşan W2 Matrisi:
[    0     a     a     1     0 a + 1]
[    a     0     1     a     0 a + 1]
[    a     1     0     a     a     a]
[    1     a     a     0 a + 1 a + 1]
[    0     0     a a + 1     0     0]
[a + 1 a + 1     a a + 